# ⚽ Highlights de Martin — Motor en Google Colab

Ejecuta las celdas con GPU. Son idempotentes: puedes volver a ejecutarlas.

> Si Colab muestra **Conectando** durante varios minutos, usa *Entorno de ejecución → Desconectar y eliminar entorno de ejecución*, vuelve a conectar y ejecuta otra vez.


In [ ]:
# 1) Comprobar la GPU
!nvidia-smi

In [ ]:
# 2) Traer o actualizar el código oficial
%cd /content
![ -d martin-highlights/.git ] && git -C martin-highlights pull --ff-only || git clone https://github.com/vjcano-gif/martin-highlights.git
%cd /content/martin-highlights


In [ ]:
# 3) Instalar dependencias (ffmpeg ya viene en Colab)
!pip install -q -r requirements.txt
!npm install -g localtunnel

## 4) Lanzar la app

⚠️ **El enlace de localtunnel es público.** No compartas la URL ni subas cookies que no estés dispuesto a usar durante esta sesión. Cierra el túnel al terminar.


In [ ]:
# Muestra la IP (contraseña del túnel)
!curl -s https://loca.lt/mytunnelpassword && echo '  ← esta es la contraseña'

In [ ]:
# Reinicia la app y el túnel sin dejar esta celda cargando indefinidamente
!pkill -f 'streamlit run app.py' || true
!pkill -f 'localtunnel --port 8501' || true
!nohup streamlit run app.py --server.headless true --server.port 8501 >/content/logs.txt 2>&1 &
!nohup npx --yes localtunnel --port 8501 >/content/tunnel.log 2>&1 &

import pathlib, re, time
tunnel_url = None
for _ in range(30):
    time.sleep(1)
    log = pathlib.Path('/content/tunnel.log').read_text(errors='replace') if pathlib.Path('/content/tunnel.log').exists() else ''
    match = re.search(r'https://[^\s]+\.loca\.lt', log)
    if match:
        tunnel_url = match.group(0)
        break
if tunnel_url:
    print(f'✅ App lista: {tunnel_url}')
    print('La app sigue activa en segundo plano; esta celda ya puede terminar.')
else:
    print('❌ El túnel no respondió en 30 segundos. Ejecuta: !cat /content/tunnel.log')


### ¿Problemas?
- La última celda ahora termina cuando obtiene el enlace; Streamlit y el túnel continúan en segundo plano.
- Si aparece **Conectando** antes de ejecutar una celda, reinicia el entorno desde el menú de Colab.
- Si YouTube bloquea la descarga ("confirma que no eres un robot"), exporta un `cookies.txt` desde tu navegador y súbelo en *Opciones avanzadas*.
- Para ver errores de la app: `!cat /content/logs.txt`. Para el túnel: `!cat /content/tunnel.log`.
